In [ ]:
# === Tutorial bootstrap: fetch utils/ + sample data if missing (for Colab blob links) ===
import os, sys, urllib.request

REPO   = "amirfar76/neurips25-valid-hparam-selection"
BRANCH = "main"
BASE   = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"

def ensure_utils():
    os.makedirs("utils", exist_ok=True)
    for fname in ["csvio.py", "testing.py"]:
        url = f"{BASE}/utils/{fname}"
        dst = os.path.join("utils", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)
    if "utils" not in sys.path:
        sys.path.append(os.path.abspath("utils"))

def ensure_data():
    os.makedirs("data", exist_ok=True)
    for fname in ["sample_binary_losses.csv", "sample_real_losses.csv"]:
        url = f"{BASE}/data/{fname}"
        dst = os.path.join("data", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)

ensure_utils()
ensure_data()
print("Bootstrap done: utils/ and data/ available.")


# B (CSV) — QLTT from CSV

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv
from utils.testing import quantile_exceedances_pvalue, holm_bonferroni

csv_path='data/sample_real_losses.csv'
tau=0.9
q_star=1.5
alpha_mtp=0.05

ids, L, cols = load_losses_csv(csv_path)
rows=[]
for i, hp in enumerate(ids):
    losses=L[i]
    p=quantile_exceedances_pvalue(losses, target_q=q_star, tau=tau)
    rows.append({'hyperparam_id':hp,'emp_tau_quantile':float(np.quantile(losses,tau)),'pval':float(p)})
df=pd.DataFrame(rows)
df['selected']=holm_bonferroni(df['pval'].values, alpha=alpha_mtp)
df.sort_values(['selected','emp_tau_quantile'], ascending=[False,True])